## accuracy comparisons


In [ ]:
import json
import re
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "runs").exists():
    project_root = project_root.parent

RUNS_DIR = project_root / "runs"
RESULTS_DIR = project_root / "results"
print("runs dir:   ", RUNS_DIR)
print("results dir:", RESULTS_DIR)

runs dir:    /Users/gorpeliates/TUDelft/RP/code/runs
results dir: /Users/gorpeliates/TUDelft/RP/code/results


## 1. BP CNN — `experiment_only_backprop_*`

CNN backprop was re-run separately after architecture changes.


In [2]:
bp_cnn_experiments = [
    "experiment_only_backprop_20260604_090527",
    "experiment_only_backprop_20260604_090543",
    "experiment_only_backprop_20260604_090556",
]

In [ ]:
rows_bp_cnn = []

for exp_name in bp_cnn_experiments:
    exp_dir = RUNS_DIR / exp_name
    for run_dir in sorted(exp_dir.iterdir()):
        if run_dir.name == "logs":
            continue
        result_file = RESULTS_DIR / (run_dir.name + ".json")
        data = json.loads(result_file.read_text())

        stem = run_dir.name
        dataset = re.search(r"_(mnist|fashionmnist|cifar10|cifar100)_", stem).group(1)
        seed = int(re.search(r"_seed(\d+)$", stem).group(1))

        rows_bp_cnn.append(
            dict(
                method="bp",
                arch="cnn",
                dataset=dataset,
                seed=seed,
                sub="bp",
                test_acc=data["bp"]["test_acc"],
            )
        )

df_bp_cnn = pd.DataFrame(rows_bp_cnn)
df_bp_cnn.pivot_table(index=["dataset"], columns="seed", values="test_acc").sort_index()

seed,1231,3845,92389
dataset,,,
cifar10,0.7296,0.7294,0.7343
cifar100,0.4286,0.4294,0.4323
fashionmnist,0.9095,0.9107,0.9114
mnist,0.9886,0.9887,0.9899


### MF+AD + BP MLP — `experiment_all_*`


In [4]:
experiment_all = [
    "experiment_all_20260528_114458",
    "experiment_all_20260528_114510",
    "experiment_all_20260528_114532",
]

In [ ]:
rows_all = []

for exp_name in experiment_all:
    exp_dir = RUNS_DIR / exp_name
    for run_dir in sorted(exp_dir.iterdir()):
        if run_dir.name == "logs":
            continue
        stem = run_dir.name

        # skip dd_* runs — MF+DD is taken from experiment_only_dd
        if stem.startswith("dd_"):
            continue

        result_file = RESULTS_DIR / (stem + ".json")
        data = json.loads(result_file.read_text())

        arch = re.search(r"_(cnn|mlp)_", stem).group(1)
        dataset = re.search(r"_(mnist|fashionmnist|cifar10|cifar100)_", stem).group(1)
        seed = int(re.search(r"_seed(\d+)$", stem).group(1))

        # BP (MLP only — CNN BP was re-run separately)
        if arch == "mlp":
            rows_all.append(
                dict(
                    method="bp",
                    arch=arch,
                    dataset=dataset,
                    seed=seed,
                    sub="bp",
                    test_acc=data["bp"]["bp"]["test_acc"],
                )
            )
        # MF+AD (cumulative-prediction and final-layer prediction mode)
        rows_all.append(
            dict(
                method="mf_ad",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="ff",
                test_acc=data["autodiff"]["mono_ff"]["test_acc"],
            )
        )
        rows_all.append(
            dict(
                method="mf_ad",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="bp",
                test_acc=data["autodiff"]["mono_bp"]["test_acc"],
            )
        )

df_all = pd.DataFrame(rows_all)
df_all.pivot_table(index=["arch", "method", "sub", "dataset"], columns="seed", values="test_acc").sort_index()

seed                          155      200      1231123
arch method sub dataset                                
cnn  mf_ad  bp  cifar10        0.5359   0.5466   0.5375
                cifar100       0.2678   0.2700   0.2658
                fashionmnist   0.8897   0.8905   0.8839
                mnist          0.9725   0.9710   0.9702
            ff  cifar10        0.5463   0.5590   0.5498
                cifar100       0.2735   0.2772   0.2718
                fashionmnist   0.8860   0.8834   0.8787
                mnist          0.9663   0.9652   0.9648
mlp  bp     bp  cifar10        0.4910   0.4855   0.4830
                cifar100       0.2065   0.2115   0.2046
                fashionmnist   0.8801   0.8745   0.8820
                mnist          0.9731   0.9716   0.9731
     mf_ad  bp  cifar10        0.5202   0.5219   0.5256
                cifar100       0.1481   0.1475   0.1518
                fashionmnist   0.8606   0.8536   0.8592
                mnist          0.9467   0.9474   0.9490
            ff  cifar10        0.5212   0.5271   0.5289
                cifar100       0.1887   0.1864   0.1919
                fashionmnist   0.8623   0.8567   0.8610
                mnist          0.9488   0.9514   0.9500

## MF+DD`experiment_only_dd_*`

Separate MF+DD P=4 re-runs.


In [6]:
experiment_only_dd = [
    "experiment_only_dd_20260614_090030",
    "experiment_only_dd_20260614_090031",
]

In [ ]:
rows_dd = []

for exp_name in experiment_only_dd:
    exp_dir = RUNS_DIR / exp_name
    for run_dir in sorted(exp_dir.iterdir()):
        if run_dir.name == "logs":
            continue
        result_file = RESULTS_DIR / (run_dir.name + ".json")
        data = json.loads(result_file.read_text())

        stem = run_dir.name
        arch = re.search(r"_(cnn|mlp)_", stem).group(1)
        dataset = re.search(r"_(mnist|fashionmnist|cifar10|cifar100)_", stem).group(1)
        seed = int(re.search(r"_seed(\d+)$", stem).group(1))

        rows_dd.append(
            dict(
                method="mf_dd",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="ff",
                test_acc=data["mono_ff"]["test_acc"],
            )
        )
        rows_dd.append(
            dict(
                method="mf_dd",
                arch=arch,
                dataset=dataset,
                seed=seed,
                sub="bp",
                test_acc=data["mono_bp"]["test_acc"],
            )
        )

df_dd = pd.DataFrame(rows_dd)
df_dd.pivot_table(index=["arch", "sub", "dataset"], columns="seed", values="test_acc").sort_index()

seed                      42      123     456
arch sub dataset                             
cnn  bp  cifar10       0.3874  0.3818  0.3906
         cifar100      0.1562  0.1487  0.1491
         fashionmnist  0.8268  0.8241  0.8305
         mnist         0.9594  0.9543  0.9544
     ff  cifar10       0.4103  0.4165  0.4278
         cifar100      0.1792  0.1706  0.1795
         fashionmnist  0.8439  0.8367  0.8446
         mnist         0.9502  0.9471  0.9503
mlp  bp  cifar10       0.2997  0.3013  0.3004
         cifar100      0.0669  0.0700  0.0692
         fashionmnist  0.8208  0.8203  0.8206
         mnist         0.9326  0.9317  0.9323
     ff  cifar10       0.3765  0.3772  0.3744
         cifar100      0.1101  0.1046  0.1134
         fashionmnist  0.8347  0.8389  0.8379
         mnist         0.9323  0.9323  0.9359

In [8]:
raw = pd.concat([df_bp_cnn, df_all, df_dd], ignore_index=True)
print(f"Total rows: {len(raw)}")
raw.groupby(["arch", "method", "sub", "dataset"])["seed"].apply(list).sort_index()

Total rows: 120


arch  method  sub  dataset     
cnn   bp      bp   cifar10         [1231, 3845, 92389]
                   cifar100        [1231, 3845, 92389]
                   fashionmnist    [1231, 3845, 92389]
                   mnist           [1231, 3845, 92389]
      mf_ad   bp   cifar10         [155, 200, 1231123]
                   cifar100        [155, 200, 1231123]
                   fashionmnist    [155, 200, 1231123]
                   mnist           [155, 200, 1231123]
              ff   cifar10         [155, 200, 1231123]
                   cifar100        [155, 200, 1231123]
                   fashionmnist    [155, 200, 1231123]
                   mnist           [155, 200, 1231123]
      mf_dd   bp   cifar10              [42, 123, 456]
                   cifar100             [42, 123, 456]
                   fashionmnist         [42, 123, 456]
                   mnist                [42, 123, 456]
              ff   cifar10              [42, 123, 456]
                   cifar100      

## 5. Aggregate and display tables


In [ ]:
agg = raw.groupby(["arch", "method", "sub", "dataset"])["test_acc"].agg(mean="mean", std="std").reset_index()

DATASETS = ["mnist", "fashionmnist", "cifar10", "cifar100"]
DATASET_LABELS = {
    "mnist": "MNIST",
    "fashionmnist": "FashionMNIST",
    "cifar10": "CIFAR-10",
    "cifar100": "CIFAR-100",
}


def fmt(mean: float, std: float) -> str:
    return f"{mean:.3f} ± {std:.3f}"


def get(arch: str, method: str, dataset: str, sub: str):
    mask = (agg["arch"] == arch) & (agg["method"] == method) & (agg["dataset"] == dataset) & (agg["sub"] == sub)
    row = agg[mask]
    return row["mean"].item(), row["std"].item()


def make_table(arch: str) -> pd.DataFrame:
    records = []
    for ds in DATASETS:
        bp_mean, bp_std = get(arch, "bp", ds, sub="bp")
        ad_ff_mean, ad_ff_std = get(arch, "mf_ad", ds, sub="ff")
        ad_bp_mean, ad_bp_std = get(arch, "mf_ad", ds, sub="bp")
        dd_ff_mean, dd_ff_std = get(arch, "mf_dd", ds, sub="ff")
        dd_bp_mean, dd_bp_std = get(arch, "mf_dd", ds, sub="bp")
        records.append(
            {
                "Dataset": DATASET_LABELS[ds],
                "BP": fmt(bp_mean, bp_std),
                "MF+AD (FF)": fmt(ad_ff_mean, ad_ff_std),
                "MF+AD (BP)": fmt(ad_bp_mean, ad_bp_std),
                "MF+DD (FF)": fmt(dd_ff_mean, dd_ff_std),
                "MF+DD (BP)": fmt(dd_bp_mean, dd_bp_std),
                "Delta FF": f"{dd_ff_mean - ad_ff_mean:+.3f}",
                "Delta BP": f"{dd_bp_mean - ad_bp_mean:+.3f}",
            }
        )
    return pd.DataFrame(records).set_index("Dataset")


pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 220)

print("MLP - Test")
display(make_table("mlp"))

print()

print("CNN - Test")
display(make_table("cnn"))

MLP - Test


,BP,MF+AD (FF),MF+AD (BP),MF+DD (FF),MF+DD (BP),Delta FF,Delta BP
Dataset,,,,,,,
MNIST,0.973 ± 0.001,0.950 ± 0.001,0.948 ± 0.001,0.933 ± 0.002,0.932 ± 0.000,-0.017,-0.015
FashionMNIST,0.879 ± 0.004,0.860 ± 0.003,0.858 ± 0.004,0.837 ± 0.002,0.821 ± 0.000,-0.023,-0.037
CIFAR-10,0.486 ± 0.004,0.526 ± 0.004,0.523 ± 0.003,0.376 ± 0.001,0.300 ± 0.001,-0.150,-0.222
CIFAR-100,0.208 ± 0.004,0.189 ± 0.003,0.149 ± 0.002,0.109 ± 0.004,0.069 ± 0.002,-0.080,-0.080



CNN - Test


,BP,MF+AD (FF),MF+AD (BP),MF+DD (FF),MF+DD (BP),Delta FF,Delta BP
Dataset,,,,,,,
MNIST,0.989 ± 0.001,0.965 ± 0.001,0.971 ± 0.001,0.949 ± 0.002,0.956 ± 0.003,-0.016,-0.015
FashionMNIST,0.911 ± 0.001,0.883 ± 0.004,0.888 ± 0.004,0.842 ± 0.004,0.827 ± 0.003,-0.041,-0.061
CIFAR-10,0.731 ± 0.003,0.552 ± 0.007,0.540 ± 0.006,0.418 ± 0.009,0.387 ± 0.004,-0.134,-0.153
CIFAR-100,0.430 ± 0.002,0.274 ± 0.003,0.268 ± 0.002,0.176 ± 0.005,0.151 ± 0.004,-0.098,-0.117
